# 🎯 Technique 84: Automatic Evaluation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/amerob/ultimate-prompt-engineering-playbook/blob/main/notebooks/10-optimization/84_automatic_evaluation.ipynb)

**Category:** 10 - Optimization & Auto-Tuning  
**Technique #:** 84  
**Difficulty:** Advanced

## 📋 Description

Automatic Evaluation uses LLMs and automated metrics to assess output quality without human intervention. This enables scalable evaluation of prompts and models.

**When to use:**
- Evaluating prompts at scale
- A/B testing approaches
- Monitoring production quality
- Regression testing

## 🔧 How It Works

Three main approaches:
1. Reference-based: BLEU, ROUGE, Exact Match
2. Reference-free: Perplexity, fluency scores
3. LLM-as-a-Judge: Rubric-based evaluation

## ⚙️ Setup

In [ ]:
!pip install -q openai numpy rouge-score
import openai
import numpy as np
from typing import List, Dict, Tuple
from dataclasses import dataclass
from rouge_score import rouge_scorer
from getpass import getpass
import re

In [ ]:
openai.api_key = getpass('Enter your OpenAI API key: ')

## 🛠️ Implementation: Evaluation Framework

In [ ]:
@dataclass
class EvalResult:
    input_text: str
    output: str
    scores: Dict[str, float]

class AutoEvaluator:
    def __init__(self, model='gpt-4o-mini'):
        self.model = model
        self.rouge = rouge_scorer.RougeScorer(['rouge1', 'rougeL'])
    
    def call_llm(self, prompt):
        response = openai.chat.completions.create(
            model=self.model,
            messages=[{'role': 'user', 'content': prompt}],
            temperature=0
        )
        return response.choices[0].message.content.strip()
    
    def exact_match(self, pred, ref):
        return 1.0 if pred.strip().lower() == ref.strip().lower() else 0.0
    
    def contains_match(self, pred, ref):
        return 1.0 if ref.strip().lower() in pred.strip().lower() else 0.0
    
    def rouge_scores(self, pred, ref):
        scores = self.rouge.score(ref, pred)
        return {'rouge1': scores['rouge1'].fmeasure, 'rougeL': scores['rougeL'].fmeasure}
    
    def llm_judge(self, input_text, output, criteria):
        prompt = f"""Rate this output (1-5) based on: {criteria}

Input: {input_text}
Output: {output}

Respond with:
Rating: [1-5]
Reason: [brief]"""
        response = self.call_llm(prompt)
        match = re.search(r'Rating:\s*(\d+)', response)
        rating = float(match.group(1)) if match else 3.0
        return rating, response
    
    def compare_outputs(self, input_text, out_a, out_b, criteria):
        prompt = f"""Which is better? Answer A, B, or TIE.

Input: {input_text}
Output A: {out_a}
Output B: {out_b}

Criteria: {criteria}"""
        response = self.call_llm(prompt)
        if 'A' in response[:5].upper() and 'B' not in response[:5].upper():
            return 'A'
        elif 'B' in response[:5].upper():
            return 'B'
        return 'TIE'

## 💡 Basic Example: Evaluating Classification

In [ ]:
evaluator = AutoEvaluator()

test_cases = [
    {'input': 'Amazing product!', 'predicted': 'POSITIVE', 'reference': 'POSITIVE'},
    {'input': 'Terrible service.', 'predicted': 'NEGATIVE', 'reference': 'NEGATIVE'},
    {'input': 'It was okay.', 'predicted': 'POSITIVE', 'reference': 'NEUTRAL'}
]

print('Classification Evaluation\n')
correct = 0
for case in test_cases:
    em = evaluator.exact_match(case['predicted'], case['reference'])
    correct += em
    status = '✓' if em else '✗'
    print(f"{status} {case['input']} -> {case['predicted']} (expected: {case['reference']})")

print(f"\nAccuracy: {correct/len(test_cases):.1%}")

## 🌍 Real-World Example: LLM-as-a-Judge

In [ ]:
support_cases = [
    {
        'query': 'My order has not arrived.',
        'response_a': 'I understand. Let me check and update you within 24 hours.',
        'response_b': 'Your order is delayed. Contact shipping.'
    }
]

criteria = 'Evaluate empathy, helpfulness, and professionalism.'

print('LLM-as-a-Judge Evaluation\n')
for case in support_cases:
    print(f"Query: {case['query']}")
    winner = evaluator.compare_outputs(case['query'], case['response_a'], case['response_b'], criteria)
    print(f"Winner: Response {winner}\n")

## ⚠️ Failure Case: Evaluation Limitations

In [ ]:
print('''Auto-Evaluation Limitations:

1. Reference-based: Require ground truth
2. LLM-as-Judge: Can be biased, expensive
3. Hallucination in evaluation
4. May miss domain-specific errors

Best Practices:
- Combine multiple methods
- Validate against human judgments
- Monitor for evaluation drift''')

## 📊 Evaluation Benchmarks

| Method | Pros | Cons | Best For |
|--------|------|------|----------|
| Exact Match | Simple | Inflexible | Classification |
| ROUGE | Captures overlap | Surface-level | Summarization |
| LLM Judge | Flexible | Expensive | Open-ended |

## 🎮 Interactive Playground

In [ ]:
YOUR_INPUT = 'Your input text'
YOUR_OUTPUT = 'Your generated output'
YOUR_CRITERIA = 'Your evaluation criteria'

# my_eval = AutoEvaluator()
# score, reason = my_eval.llm_judge(YOUR_INPUT, YOUR_OUTPUT, YOUR_CRITERIA)
# print(f'Score: {score}/5')

## 💡 Tips & Tricks

- Use cheaper models for initial screening
- Batch evaluations when possible
- Cache evaluation results
- Combine multiple metrics

## 📚 References

1. [LLM-as-a-Judge](https://arxiv.org/abs/2306.05685) - Zheng et al.
2. [ROUGE Scoring](https://aclanthology.org/W04-1013/)
3. [G-Eval](https://arxiv.org/abs/2303.16634)